# 02 - Streaming ingestion (Bonus 2)

Uses Auto Loader to incrementally read new CSV files from
`/Volumes/marathos/default/raw/streaming/` and append them to
`marathos.bronze.streaming_results`.

Auto Loader keeps track of which files it has already processed via a checkpoint,
so re-running the stream only picks up new files.

We run with `availableNow=True` so the stream processes pending files and then
stops, which suits interactive use. In production this would run continuously.

*Some parts and solutions of this code was found through help from Claude*

In [0]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

from pyspark.sql import functions as F

from utils.constants import (
    STREAMING_SOURCE_PATH,
    STREAMING_CHECKPOINT_PATH,
    BRONZE_STREAMING_RESULTS,
)

## Pre-create the target table with column mapping

The source CSV has column names with spaces and slashes, which Delta doesn't
allow by default. We create the table up front with column mapping enabled,
matching what we did for the batch bronze table.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_STREAMING_RESULTS} (
        `Year of event` INT,
        `Event dates` STRING,
        `Event name` STRING,
        `Event distance/length` STRING,
        `Event number of finishers` INT,
        `Athlete performance` STRING,
        `Athlete club` STRING,
        `Athlete country` STRING,
        `Athlete year of birth` DOUBLE,
        `Athlete gender` STRING,
        `Athlete age category` STRING,
        `Athlete average speed` STRING,
        `Athlete ID` INT
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5'
    )
""")
print(f"Created {BRONZE_STREAMING_RESULTS}")

## Define the stream

Reads from the streaming folder using Auto Loader (cloudFiles).
Each new CSV file dropped into the folder will be picked up on the next run.

In [0]:
stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", STREAMING_CHECKPOINT_PATH)
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(STREAMING_SOURCE_PATH)
)

## Start the stream

Writes to the bronze.streaming_results table with checkpoint enabled.
`availableNow=True` makes the stream process pending files and then stop.

In [0]:
query = (
    stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", STREAMING_CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_STREAMING_RESULTS)
)

query.awaitTermination()
print(f"Stream finished. Wrote new files to {BRONZE_STREAMING_RESULTS}")

## Verify the result

Read back the streaming table and confirm rows landed.

In [0]:
streaming_table = spark.table(BRONZE_STREAMING_RESULTS)
print(f"Rows in streaming_results: {streaming_table.count():,}")
streaming_table.show(5, vertical=True, truncate=False)